# Final Project: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Course:** Python Project for Data Science  
**Estimated Time:** 1 Hour

---

In this assignment, you will extract stock data and revenue data for **Tesla** and **GameStop**, then build interactive dashboards to compare each company's stock price vs. its revenue over time.

## Table of Contents
1. [Setup & Imports](#setup)
2. [Question 1 – Extracting Tesla Stock Data Using yfinance](#q1)
3. [Question 2 – Extracting Tesla Revenue Data Using Web Scraping](#q2)
4. [Question 3 – Extracting GameStop Stock Data Using yfinance](#q3)
5. [Question 4 – Extracting GameStop Revenue Data Using Web Scraping](#q4)
6. [Question 5 – Tesla Stock and Revenue Dashboard](#q5)
7. [Question 6 – GameStop Stock and Revenue Dashboard](#q6)

## Setup & Imports <a id='setup'></a>

Install and import required libraries.

In [ ]:
# Install dependencies (run once)
!pip install yfinance==0.2.38 requests bs4 pandas plotly --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Define the Dashboard Function

We first define a reusable helper function `make_graph()` that builds an interactive two-panel Plotly dashboard showing stock price (top) and quarterly revenue (bottom).

In [ ]:
def make_graph(stock_data, revenue_data, stock_name):
    """
    Creates a two-panel interactive Plotly dashboard.
    
    Parameters
    ----------
    stock_data   : pd.DataFrame  – must contain columns 'Date' and 'Close'
    revenue_data : pd.DataFrame  – must contain columns 'Date' and 'Revenue'
    stock_name   : str           – company name displayed in the title
    """
    # ── Ensure datetime types ──────────────────────────────────────────────
    stock_data   = stock_data.copy()
    revenue_data = revenue_data.copy()
    stock_data['Date']   = pd.to_datetime(stock_data['Date'])
    revenue_data['Date'] = pd.to_datetime(revenue_data['Date'])

    # ── Clip to most-recent 10 years ───────────────────────────────────────
    cutoff = pd.Timestamp.today() - pd.DateOffset(years=10)
    stock_data   = stock_data[stock_data['Date'] >= cutoff]
    revenue_data = revenue_data[revenue_data['Date'] >= cutoff]

    # ── Clean revenue column ───────────────────────────────────────────────
    revenue_data['Revenue'] = (
        revenue_data['Revenue']
        .astype(str)
        .str.replace(r'[\$,]', '', regex=True)
        .replace('', float('nan'))
    )
    revenue_data = revenue_data.dropna(subset=['Revenue'])
    revenue_data['Revenue'] = revenue_data['Revenue'].astype(float)

    # ── Build figure ───────────────────────────────────────────────────────
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(f'{stock_name} Historical Share Price',
                        f'{stock_name} Historical Revenue'),
        shared_xaxes=True,
        vertical_spacing=0.12
    )

    # Panel 1 – Stock price
    fig.add_trace(
        go.Scatter(
            x=stock_data['Date'],
            y=stock_data['Close'].astype(float),
            mode='lines',
            name='Close Price',
            line=dict(color='royalblue', width=1.5)
        ),
        row=1, col=1
    )

    # Panel 2 – Quarterly revenue
    fig.add_trace(
        go.Bar(
            x=revenue_data['Date'],
            y=revenue_data['Revenue'],
            name='Quarterly Revenue (USD M)',
            marker_color='orange'
        ),
        row=2, col=1
    )

    # ── Layout ─────────────────────────────────────────────────────────────
    fig.update_layout(
        height=800,
        title_text=f'<b>{stock_name}</b> – Stock Price vs. Revenue Dashboard',
        title_x=0.5,
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_yaxes(title_text='Close Price (USD)', row=1, col=1)
    fig.update_yaxes(title_text='Revenue (USD Millions)', row=2, col=1)
    fig.update_xaxes(title_text='Date', row=2, col=1)

    fig.show()

print('make_graph() defined ✓')

---
## Question 1 – Extracting Tesla Stock Data Using yfinance <a id='q1'></a>

Use the `yfinance` library to extract Tesla's historical share-price data and save the result in a DataFrame called `tesla_data`.  
Reset the index so that *Date* becomes a column, then display the first five rows.

In [ ]:
# ── Question 1 ────────────────────────────────────────────────────────────

tesla_ticker = yf.Ticker('TSLA')

# Download full history (all available dates)
tesla_data = tesla_ticker.history(period='max')

# Reset index so 'Date' is a regular column
tesla_data.reset_index(inplace=True)

# Preview
print(f'Tesla stock data shape: {tesla_data.shape}')
tesla_data.head()

---
## Question 2 – Extracting Tesla Revenue Data Using Web Scraping <a id='q2'></a>

Use `requests` and `BeautifulSoup` to scrape Tesla's quarterly revenue from Macrotrends.  
Store the cleaned result in a DataFrame called `tesla_revenue` and display the **last** five rows.

In [ ]:
# ── Question 2 ────────────────────────────────────────────────────────────

url_tesla = 'https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue'

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/124.0 Safari/537.36'
    )
}

response = requests.get(url_tesla, headers=headers)
soup     = BeautifulSoup(response.text, 'html.parser')

# Macrotrends stores the data in multiple tables; grab the quarterly one
tables = pd.read_html(response.text)

# The quarterly revenue table is usually the second table (index 1)
tesla_revenue = tables[1].copy()
tesla_revenue.columns = ['Date', 'Revenue']

# Clean: remove dollar signs, commas, and NaN rows
tesla_revenue['Revenue'] = (
    tesla_revenue['Revenue']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
)
tesla_revenue.dropna(subset=['Revenue'], inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != '']
tesla_revenue['Date'] = pd.to_datetime(tesla_revenue['Date'])

print(f'Tesla revenue data shape: {tesla_revenue.shape}')
tesla_revenue.tail()

---
## Question 3 – Extracting GameStop Stock Data Using yfinance <a id='q3'></a>

Use `yfinance` to extract GameStop's historical share-price data.  
Store the result in `gme_data`, reset the index, and display the first five rows.

In [ ]:
# ── Question 3 ────────────────────────────────────────────────────────────

gme_ticker = yf.Ticker('GME')

gme_data = gme_ticker.history(period='max')
gme_data.reset_index(inplace=True)

print(f'GameStop stock data shape: {gme_data.shape}')
gme_data.head()

---
## Question 4 – Extracting GameStop Revenue Data Using Web Scraping <a id='q4'></a>

Scrape GameStop's quarterly revenue from Macrotrends and save it to `gme_revenue`.  
Display the **last** five rows.

In [ ]:
# ── Question 4 ────────────────────────────────────────────────────────────

url_gme = 'https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue'

response_gme = requests.get(url_gme, headers=headers)
tables_gme   = pd.read_html(response_gme.text)

gme_revenue = tables_gme[1].copy()
gme_revenue.columns = ['Date', 'Revenue']

gme_revenue['Revenue'] = (
    gme_revenue['Revenue']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
)
gme_revenue.dropna(subset=['Revenue'], inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != '']
gme_revenue['Date'] = pd.to_datetime(gme_revenue['Date'])

print(f'GameStop revenue data shape: {gme_revenue.shape}')
gme_revenue.tail()

---
## Question 5 – Tesla Stock and Revenue Dashboard <a id='q5'></a>

Use `make_graph()` to produce the Tesla dashboard.  
The **top panel** shows the historical closing price; the **bottom panel** shows the quarterly revenue.

In [ ]:
# ── Question 5 ────────────────────────────────────────────────────────────
make_graph(tesla_data, tesla_revenue, 'Tesla')

---
## Question 6 – GameStop Stock and Revenue Dashboard <a id='q6'></a>

Use `make_graph()` to produce the GameStop dashboard.

In [ ]:
# ── Question 6 ────────────────────────────────────────────────────────────
make_graph(gme_data, gme_revenue, 'GameStop')

---
## Summary

| # | Task | Library / Method |
|---|------|-----------------|
| Q1 | Tesla stock history | `yfinance` |
| Q2 | Tesla quarterly revenue | `requests` + `BeautifulSoup` + `pd.read_html` |
| Q3 | GameStop stock history | `yfinance` |
| Q4 | GameStop quarterly revenue | `requests` + `BeautifulSoup` + `pd.read_html` |
| Q5 | Tesla dashboard | `plotly` (`make_graph`) |
| Q6 | GameStop dashboard | `plotly` (`make_graph`) |

**To submit:** Take a screenshot of each output (including question header + result) and upload to Coursera.